# Лекция 7. Лица: PCA и кластеризация — живой ноутбук

Ноутбук для демонстрации на лекции. Верхний блок «Пульт лектора» задаёт параметры;
меняйте значения и перезапускайте нужные ячейки.

Набор Labeled Faces in the Wild описан в учебнике на с. 164; пример кластеризации
лиц — на с. 213.

Порядок: данные → PCA → собственные лица → восстановление → k-средних →
агломеративная → DBSCAN → сравнение.


## Запуск из GitHub

Ноутбук можно открыть в Colab или Binder, если репозиторий загружен на GitHub.

- Colab: `https://colab.research.google.com/github/aaguberniev/lect17/blob/main/plans/notebooks/lecture7_faces_pca_clustering.ipynb`
- Binder: `https://mybinder.org/v2/gh/aaguberniev/lect17/HEAD?labpath=plans/notebooks/lecture7_faces_pca_clustering.ipynb`
- Codespaces: Code → Codespaces → Open, затем `pip install -r requirements.txt`.

Замените `OWNER/REPO` на свой репозиторий. GitHub сам ноутбук не исполняет —
он только показывает его; для запуска нужен Colab, Binder или Codespaces.


In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn", "matplotlib", "scipy"], check=True)
print("Colab:", IN_COLAB)


## 0. Подготовка

При первом запуске `fetch_lfw_people` скачает Labeled Faces in the Wild
(около 200 МБ) и сохранит в `~/scikit_learn_data`. Дальше ноутбук работает из кеша.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_lfw_people
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score, silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["image.cmap"] = "gray"
np.random.seed(0)


## 1. Пульт лектора

Меняйте эти числа и перезапускайте ячейки ниже.

In [ ]:
N_COMPONENTS = 100      # сколько компонент PCA оставить
N_CLUSTERS = 10         # число кластеров для k-средних и агломеративной
EPS = 7.0               # радиус DBSCAN
MIN_SAMPLES = 3         # порог плотности DBSCAN
N_SHOW = 10             # сколько лиц показывать в ряду
RANDOM_STATE = 0


## 2. Загрузка и осмотр данных

In [ ]:
people = fetch_lfw_people(min_faces_per_person=20, resize=0.7)
X = people.data
y = people.target
image_shape = people.images[0].shape

print("изображения:", people.images.shape)
print("признаки:", X.shape)
print("людей:", len(people.target_names))
print("размер лица:", image_shape)


In [ ]:
def show_faces(images, titles=None, ncols=10, scale=1.35):
    images = np.asarray(images)
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * scale, nrows * scale * 1.15))
    axes = np.atleast_2d(axes)
    for i, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i])
            if titles is not None:
                ax.set_title(str(titles[i]), fontsize=8)
    plt.tight_layout()
    plt.show()


def show_cluster_examples(labels, n=8):
    labels = np.asarray(labels)
    clusters = [c for c in np.unique(labels) if c != -1]
    for c in clusters:
        idx = np.where(labels == c)[0][:n]
        print("кластер", c, "размер", int(np.sum(labels == c)))
        show_faces(people.images[idx], ncols=n)


In [ ]:
show_faces(people.images[:N_SHOW], titles=[people.target_names[t].split()[-1] for t in y[:N_SHOW]], ncols=N_SHOW)


## 3. PCA: сжатие лиц

Меняйте `N_COMPONENTS` и перезапускайте ячейки 3–5.

In [ ]:
pca = PCA(n_components=N_COMPONENTS, svd_solver="randomized", random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)

print("было признаков:", X.shape[1])
print("стало компонент:", X_pca.shape[1])
print("объяснённая дисперсия:", round(float(pca.explained_variance_ratio_.sum()), 3))


In [ ]:
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("число компонент")
plt.ylabel("накопленная объяснённая дисперсия")
plt.title("Сколько компонент оставить")
plt.grid(alpha=0.3)
plt.show()


## 4. Собственные лица

Компоненты PCA — направления изменения яркости.

In [ ]:
n_eigen = min(15, N_COMPONENTS)
show_faces(pca.components_[:n_eigen].reshape(-1, *image_shape),
           titles=[f"PC{i+1}" for i in range(n_eigen)])


## 5. Восстановление лица

Меняйте `KS` — список чисел компонент.

In [ ]:
KS = [10, 50, 100, 300]
max_k = max(KS)

pca_full = PCA(n_components=min(max_k, X.shape[1]), svd_solver="randomized",
               random_state=RANDOM_STATE).fit(X)
Z_full = pca_full.transform(X)


def reconstruct(index, k):
    z = Z_full[index:index + 1].copy()
    z[:, k:] = 0
    return pca_full.inverse_transform(z).reshape(image_shape)


def show_reconstructions(index):
    imgs = [X[index].reshape(image_shape)]
    titles = ["оригинал"]
    for k in KS:
        imgs.append(reconstruct(index, k))
        titles.append(f"{k} компонент")
    show_faces(imgs, titles=titles, ncols=len(imgs), scale=2.0)


show_reconstructions(0)


## 6. K-средних на лицах

Меняйте `N_CLUSTERS` и перезапускайте ячейки 6–7.

In [ ]:
km = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_STATE)
labels_km = km.fit_predict(X_pca)

counts = np.bincount(labels_km, minlength=N_CLUSTERS)
print("размеры кластеров:", counts)

centers = pca.inverse_transform(km.cluster_centers_).reshape(-1, *image_shape)
show_faces(centers, titles=[f"cluster {i}: {counts[i]}" for i in range(N_CLUSTERS)])


In [ ]:
show_cluster_examples(labels_km, n=N_SHOW)


## 7. Агломеративная кластеризация

In [ ]:
agg = AgglomerativeClustering(n_clusters=N_CLUSTERS)
labels_agg = agg.fit_predict(X_pca)
print("размеры кластеров:", np.bincount(labels_agg, minlength=N_CLUSTERS))


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(X_pca), size=300, replace=False)
Z = linkage(X_pca[idx], method="ward")

plt.figure(figsize=(14, 4))
dendrogram(Z, truncate_mode="level", p=7, no_labels=True)
plt.xlabel("объекты")
plt.ylabel("расстояние объединения")
plt.title("Дендрограмма по 300 случайным лицам")
plt.show()


In [ ]:
show_cluster_examples(labels_agg, n=N_SHOW)


In [ ]:
agg40 = AgglomerativeClustering(n_clusters=40).fit_predict(X_pca)
sizes40 = np.bincount(agg40, minlength=40)
print("40 кластеров: минимальный размер", sizes40.min(), "максимальный", sizes40.max())


## 8. DBSCAN

Меняйте `EPS` и `MIN_SAMPLES`; смотрите, сколько лиц стало шумом.

In [ ]:
db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
labels_db = db.fit_predict(X_pca)

unique = np.unique(labels_db)
n_clusters_db = len(unique) - (1 if -1 in unique else 0)
n_noise = int(np.sum(labels_db == -1))
print("уникальные метки:", unique)
print("кластеров:", n_clusters_db, "шум:", n_noise)


In [ ]:
show_cluster_examples(labels_db, n=N_SHOW)


### Перебор eps

Одна ячейка — несколько запусков, чтобы показать чувствительность.

In [ ]:
for eps in [3, 5, 7, 9, 11, 13, 15]:
    labels = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit_predict(X_pca)
    u = np.unique(labels)
    k = len(u) - (1 if -1 in u else 0)
    noise = int(np.sum(labels == -1))
    print(f"eps={eps:>4}: кластеров {k:>2}, шум {noise:>4}")


## 9. Сравнение разбиений

In [ ]:
print("ARI k-средних и агломеративной:", round(adjusted_rand_score(labels_km, labels_agg), 3))
print("ARI k-средних и DBSCAN:", round(adjusted_rand_score(labels_km, labels_db), 3))
print("Силуэт k-средних:", round(silhouette_score(X_pca, labels_km), 3))


## 10. Что менять на лекции

- `N_COMPONENTS`: 10 → 50 → 100 → 300. Показать, как меняются собственные лица и восстановление.
- `N_CLUSTERS`: 5 → 10 → 20. Показать, что центры k-средних смешивают людей.
- `EPS`: 3 → 7 → 15. Показать рост/слияние кластеров и долю шума.
- `MIN_SAMPLES`: 2 → 3 → 10. Показать чувствительность DBSCAN.
- `KS`: 10, 50, 100, 300. Показать качество восстановления лица.
- `N_SHOW`: сколько лиц показывать в ряду.

Числа кластеров и состав групп могут отличаться от книги: версия scikit-learn и отсутствие дополнительного
масштабирования компонент влияют на пространство признаков. Смотрите на поведение алгоритма, а не на
точное совпадение числа кластеров.

После каждого изменения перезапускайте ячейку и следующую за ней; остальное можно не трогать.
